# Overfitting in the Sandbox

**Week 2 · Lesson 9** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week2/lectures/09_overfitting_in_the_sandbox.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 2.2 ("Assessing Model Accuracy") where relevant.*

## Setup

Same boilerplate as every notebook this term — see Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Why this matters:** Lesson 8's curve — training error falling forever,
test error turning back upward — is easy to nod along to and easy to miss
in practice, because a real project only ever sees one finite, noisy
sample. A model that fits *that* sample beautifully can still be quietly
memorizing its noise rather than its signal, and nothing about the training
number will tell you which one happened. This notebook makes the mechanism
undeniable: fit the same data with increasingly flexible polynomials and
watch training error fall monotonically while a *held-out* sample reveals
the U-shape underneath.

## 2. Data

`make_regression_sandbox` (`gbus564_coursework.utils.sandbox`, toured in Week 1 Lesson 9)
with `n_features=1` returns points drawn from a fixed cubic signal plus
Gaussian noise. One deliberate cheat, only possible in a sandbox: a real
project gets one finite sample to split into train/test, but here we can
draw a **second, much larger** independent sample from the same process to
stand in for "the truth" — a near-noise-free readout of test error, instead
of the noisy estimate a small real test set would give.

The dashed line in the plot below is that fixed signal, $y = 0.5x^3 - 2x$ —
the same process run with `noise=0` — drawn for reference only. A real project never gets to see it; every model
that follows has to work from the 25 points alone.

In [ ]:
from gbus564_coursework.utils.sandbox import make_regression_sandbox

# Small, noisy "observed" sample -- what we'd actually have in practice.
X_train, y_train = make_regression_sandbox(n_samples=25, n_features=1, noise=1.5, rng=rng)
# Large independent sample from the same process -- a sandbox-only stand-in
# for "the truth," used purely to see the test curve cleanly.
X_test, y_test = make_regression_sandbox(n_samples=1000, n_features=1, noise=1.5, rng=rng)
# Same process with noise=0 -- the true signal itself, only knowable in a
# sandbox. Drawn last so it doesn't shift the rng for the two samples above.
X_true, y_true = make_regression_sandbox(n_samples=1000, n_features=1, noise=0, rng=rng)
order = X_true["x"].argsort()

fig, ax = plt.subplots()
ax.scatter(X_train["x"], y_train, color="#1A73E8", label="observed sample (n=25)")
ax.plot(X_true["x"].iloc[order], y_true.iloc[order], color="black", lw=1.8, linestyle="--",
        label="true signal (unknown in practice)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("The 25 points a real project would actually see")
ax.legend()
plt.tight_layout()

## 3. Model

The baseline is the simplest polynomial — degree 1, plain linear
regression. The model under study is the same linear regression, refit on
increasingly higher-degree polynomial features of `x`, so "more flexible"
means nothing more exotic than adding `x^2, x^3, ..., x^d` as extra
columns.

There's no separate "polynomial regression" algorithm. Polynomial
regression *is* linear regression on polynomial columns — the model is still
a weighted sum, $b_0 + b_1 x + b_2 x^2 + \dots$, and the curve comes entirely
from the features. It's the same transformer → estimator `Pipeline` from
Lesson 8, with `PolynomialFeatures` in place of the encoder. Here's what it
actually hands to `LinearRegression` at degree 3:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# What PolynomialFeatures actually hands to LinearRegression: the same x,
# plus its powers as extra columns. The "polynomial" lives here, not in the model.
poly3 = PolynomialFeatures(degree=3, include_bias=False)
pd.DataFrame(poly3.fit_transform(X_train.head()), columns=poly3.get_feature_names_out())

### Model Selection

Sweep degree 1 through 15 rather than picking one — the whole point of this
notebook is watching the *sweep*, not landing on a single winning degree.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

degrees = range(1, 16)
train_mse, test_mse, fitted_models = [], [], {}

for d in degrees:
    model = make_pipeline(PolynomialFeatures(degree=d, include_bias=False), LinearRegression())
    model.fit(X_train, y_train)
    fitted_models[d] = model
    train_mse.append(mean_squared_error(y_train, model.predict(X_train)))
    test_mse.append(mean_squared_error(y_test, model.predict(X_test)))

baseline = LinearRegression().fit(X_train, y_train)
baseline_test_mse = mean_squared_error(y_test, baseline.predict(X_test))
print(f"Baseline (degree=1) test MSE: {baseline_test_mse:.2f}")

## 4. Evaluate

*Score against the baseline, not in isolation — a metric only means
something next to what it's beating.*

Lesson 8's curve, reproduced from data instead of drawn by hand: training
MSE falls the entire way as degree increases, while test MSE — measured on
the large independent sample — is U-shaped, with a clear best degree in the
middle.

In [ ]:
best_degree = degrees[int(np.argmin(test_mse))]
print(f"Best degree by held-out test MSE: {best_degree}")

fig, ax = plt.subplots()
ax.plot(list(degrees), train_mse, marker="o", color="#1A73E8", label="Training MSE")
ax.plot(list(degrees), test_mse, marker="o", color="#B33A3A", label="Test MSE")
ax.axvline(best_degree, color="#9AA0A6", linestyle="--", label=f"best degree = {best_degree}")
ax.set_yscale("log")  # high degrees blow up by orders of magnitude -- log
                       # scale keeps the low-degree U-shape from flattening out
ax.set_xlabel("Polynomial degree (flexibility)")
ax.set_ylabel("Mean squared error (log scale)")
ax.set_title("Training error keeps falling; test error is U-shaped")
ax.legend()
plt.tight_layout()

**A tie, not a win.** The true signal is cubic, so why does degree 4 come out
on top? Look at the numbers. Degree 3's test MSE is 2.245 and degree 4's is
2.231. That's a 0.6% gap, and both sit right at the noise floor: the sandbox
adds noise with standard deviation 1.5, so even the true curve scores about
1.5² = 2.25. Degree 4's extra $x^4$ term gets a weight of −0.016, effectively
zero. It "wins" by fitting this test sample's noise slightly better. Rerun
with a different seed and degree 3 usually wins instead. The practical rule:
when two models are this close, pick the simpler one.

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

Three fitted curves against the 25 observed points: an underfit degree that
misses the bend in the data, a near-best degree that tracks the signal, and
a high degree that snakes through every individual noisy point — chasing
noise instead of the cubic signal underneath.

In [ ]:
illustrative_degrees = [1, best_degree, 15]
x_grid = np.linspace(X_train["x"].min(), X_train["x"].max(), 300).reshape(-1, 1)
x_grid_df = pd.DataFrame({"x": x_grid.ravel()})
# Fix the y-range to the data's own scale -- the degree=15 fit swings wildly
# near the edges (classic high-degree-polynomial instability), and letting
# that set a shared axis would flatten the other two panels to invisible.
y_lo, y_hi = y_train.min() - 5, y_train.max() + 5

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
for ax, d in zip(axes, illustrative_degrees):
    ax.scatter(X_train["x"], y_train, color="#1A73E8", s=25, alpha=0.7, label="observed")
    ax.plot(x_grid_df["x"], fitted_models[d].predict(x_grid_df), color="#B33A3A", lw=2.2)
    label = "underfit" if d == 1 else ("overfit" if d == 15 else "near-best")
    ax.set_title(f"degree={d}  ({label})")
    ax.set_xlabel("x")
    ax.set_ylim(y_lo, y_hi)
axes[0].set_ylabel("y")
axes[0].legend(fontsize=8)
plt.tight_layout()

## 6. Recommend

**The rule:** never judge model flexibility by how well it fits the data
you can see — a sufficiently flexible model will always improve that
number, right up to memorizing the noise. Judge flexibility by error on
data it *didn't* train on, and expect a U-shape, not a straight line, as
flexibility increases. In practice that held-out data is k-fold CV (Lesson
2), not a giant independent sample like this notebook's sandbox-only cheat.

**Caveats / next steps:** this notebook picked the best degree by looking
at test MSE directly — fine for a demo, but in a real project you'd never
touch the test set to make that choice (Lesson 10's job). And "add
polynomial terms" is only one of several complexity knobs — Weeks 3, 5, and
6 each supply a different one (shrinkage, tree depth, dropout) for the
model families that week introduces.